# 4 Co-creating an image-analysis function with an LLM
Co-create a reusable measurement of protein fluorescence near cilia, using the two images in this directory.

- **Copy–modify–create:** build on the base case, then adapt it for reuse.
- **Predict–code–reflect:** anticipate results, run the analysis, and interpret the evidence.
- **Initiate–refine:** ask the LLM for a proposal, review it, and give specific feedback before accepting it.

Keep your accepted plan, prompts, and important revisions so a colleague can follow your decisions.

This solution contains author-written examples, not a recorded LLM interaction.

## A “brain-rot” warning: keep mental ownership
An LLM can distance you from the solution and encourage passive acceptance. **You remain responsible for the analysis and its quality.**

- Decide what the measurement must achieve before accepting a method.
- Account for the processing steps in your own words.
- Check results independently and investigate surprises.
- Identify what you still need to understand before reusing or changing the code.

## 4.1 Base case: protein intensity near cilia
- `Image_34362.tif`: cilia, mainly in the green channel.
- `Image_34363.tif`: protein signal, mainly in the red channel.
- The images must have matching dimensions **and corresponding pixel positions**.
- RGB channel indices are 0 = red, 1 = green, 2 = blue; `[:,:,1]` selects the green channel.

Run the notebook with `4_LLM` as the working directory.

**Predict:** Which channel should show each signal? Why must their pixel positions correspond?

In [ ]:
from skimage import io, morphology
import numpy as np
from matplotlib import pyplot as plt
plt.rcParams['image.cmap'] = 'gray'

# Read the two images included alongside this notebook.
cilia = io.imread('Image_34362.tif')
protein = io.imread('Image_34363.tif')
green = cilia[:,:,1]
red = protein[:,:,0]
plt.imshow(cilia)
plt.title('Cilia')
plt.show()
plt.imshow(protein)
plt.title('Protein')
plt.show()


**Reflect:** Do the displayed signals match those expectations? What would a misalignment mean for the measurement?

The base case:

1. Select cilia pixels with green intensity above a threshold.
2. Expand their mask by a disk radius in pixels, including the cilia themselves.
3. Report protein intensity sum, selected area, and mean intensity. An empty mask has an undefined mean (`np.nan`).

Threshold 75 and radius 20 are example settings. This direct calculation supplies reference results for the reusable function.

**Predict:** If the radius increases, what should happen to area and total intensity? Must the mean increase?

In [ ]:
# Direct base-case calculation with explicit threshold and radius choices.
chosenThreshold = 75
chosenRadius = 20
segm = green > chosenThreshold
msk = morphology.dilation(segm, morphology.disk(chosenRadius))
redMsk = red.astype(float) * msk
referenceTotal = redMsk.sum()
referenceArea = msk.sum()
if referenceArea == 0:
    referenceMean = np.nan
else:
    referenceMean = referenceTotal / referenceArea
print('Reference intensity sum:', referenceTotal)
print('Reference mask area (pixels):', referenceArea)
print('Reference mean intensity:', referenceMean)
plt.imshow(redMsk)
plt.title('Protein intensities inside the expanded cilia mask')
plt.show()


**Reflect:** What do the three results mean? Why is fluorescence intensity not automatically an absolute amount of protein?

## 4.2 Plan the function: requirements before methods
Keep these three levels distinct:

| Level | Question | Example |
|---|---|---|
| **Requirements** | What must the analysis achieve? | A colleague can repeat the same measurement on another matched image pair. |
| **Design** | Which approach meets that need, and why? | Supply images and settings as function inputs so reuse needs no edits to its body. |
| **Code specification** | What exact interface and behaviour should be implemented? | Argument names, return order, and error handling. Develop this when requesting code in 4c. |

**Start with the scientific need:** what “near” means, the quantities and units required, assumptions, and limits. A list of processing operations alone does not explain the requirements or justify the design.

### Prompt, review, refine
Use this approach throughout:

1. Request an initial proposal or give a draft for review.
2. Check it against your scientific intent; identify a missing decision or unsupported assumption.
3. Give specific feedback, inspect the revision, and save what you accept.

For example, start with:

```text
We need to quantify protein fluorescence near cilia in paired green/red images. Help us clarify the requirements: purpose, meaning of “near”, required outputs and units, assumptions, and reuse needs. Identify unresolved scientific decisions. Do not propose processing steps or code yet.
```

After agreeing on requirements, ask:

```text
Given these accepted requirements [paste them], suggest a simple design. Justify the main choices, discuss an alternative and its tradeoff, and identify assumptions. Do not write code yet.
```

For either stage, refine with: “Our review found [issue]. We intend [decision] because [reason]. Revise accordingly.”

**Assignment 4a:** Co-create the requirements first. Then agree on a design, recording why its main choices meet those requirements. Leave code details for 4c.

**Predict:** What could a technically correct program still misunderstand about your intended measurement?

**Example agreed plan**

**Requirements**

- Summarise protein fluorescence within the cilia and their chosen neighbourhood, pooling all selected pixels.
- Report total intensity, area in pixels, and mean intensity. An empty region has zero sum and area, with undefined mean.
- Allow repetition with other matching image pairs and settings, without changing input images. Individual-cilium measurements and absolute protein concentration are outside scope.

**Design and reasons**

- Keep the base-case threshold-and-disk approach for comparability. Threshold and radius remain adjustable inputs.
- Use a single measurement function; keep file loading and plotting outside it for reuse.
- Use floating-point intensities for arithmetic and `np.nan` for an undefined mean. Matched positions and finite, nonnegative intensities are assumptions.
- A ring excluding the cilia is an alternative, but answers a different spatial question. Choose the inclusive neighbourhood for this task.

The exact interface and error behaviour are agreed when requesting implementation in 4c.

**Reflect:** Does your plan state the scientific need and justify the method, beyond listing operations? Which decision improved through discussion with the LLM?

## 4.3 Quality conditions before code generation
Quality conditions state the evidence needed to accept the result:

- **Numerical:** agreement with the base case and clear empty-mask behaviour.
- **Scientific:** the selected region corresponds to the intended neighbourhood.
- **Reuse:** settings, assumptions, and results are understandable to a colleague.

An expected biological trend is a reason to investigate a surprising result, not a test the result must pass.

**Assignment 4b:** Co-create quality conditions from your requirements. For each condition, identify the evidence needed to assess it.

**Predict:** Should mean protein intensity near cilia exceed the whole-image mean? What in the images supports your expectation?

**Example quality conditions**

- **Region:** inspect the cilia and expanded mask together; assess whether the chosen neighbourhood is appropriate.
- **Agreement:** with identical inputs and settings, all three outputs match section 4.1. An empty selection gives zero sum and area, and an undefined mean.
- **Interpretation:** compare local and whole-image means. Investigate surprises without requiring a particular biological trend.
- **Parameter changes:** increasing radius cannot reduce area or total intensity for nonnegative images; the mean may rise or fall.
- **Handover:** retain image/channel choices, settings, code, outputs, and assumptions.

Agreement with the base case does not validate the biological threshold or calibrate fluorescence.

**Reflect:** Which conditions establish numerical agreement, and which need scientific judgement or further evidence?

## 4.4 Implement and verify
Now turn the plan into a code specification: agree on inputs, parameter units, return values, and unusual-case behaviour. Include these and your quality conditions in the implementation prompt.

- Paste only Python code, without Markdown fences.
- Use independent checks and compare with the base case.
- Give the LLM expected and observed results when investigating a mismatch.
- Restart the kernel and run in order to check reproducibility.

**Assignment 4c:** Request a minimal implementation of your plan (4a), with calls that assess your quality conditions (4b). Run it; compare with the base case and whole-image mean, then try another radius. Review and refine as needed.

**Predict:** With the same images and settings, should all three measurements match section 4.1?

In [ ]:
# Author-written reference implementation, not a recorded response from an LLM.
def measure_near_cilia(green_image, red_image, threshold, radius):
    """Return intensity sum, mask area (pixels), and mean near the cilia.

    Images must be same-shaped, nonempty 2D NumPy arrays with finite,
    nonnegative values; threshold must be a finite number. Segment with
    green_image > threshold and expand with a disk of radius pixels.
    Inputs are unchanged. An empty mask returns (0, 0, np.nan).
    """
    if green_image.ndim != 2 or red_image.ndim != 2:
        raise ValueError('Images must be two-dimensional')
    if green_image.shape != red_image.shape:
        raise ValueError('Images must have equal shapes')
    if not isinstance(radius, (int, np.integer)) or isinstance(radius, (bool, np.bool_)) or radius < 0:
        raise ValueError('Radius must be a nonnegative integer')

    segmentation = green_image > threshold
    mask = morphology.dilation(segmentation, morphology.disk(radius))
    masked_red = red_image.astype(float) * mask
    total = masked_red.sum()
    area = mask.sum()
    if area == 0:
        mean = np.nan
    else:
        mean = total / area
    return total, area, mean

# Use the same arrays and parameter choices as the direct base-case calculation.
total, area, mean = measure_near_cilia(green, red, chosenThreshold, chosenRadius)
print('Base-case sum, area, mean:', referenceTotal, referenceArea, referenceMean)
print('Function intensity sum:', total)
print('Function mask area (pixels):', area)
print('Function mean intensity:', mean)
# Compare the printed function results with the base-case values above.

# Compare with the whole protein image, as predicted in section 4.3.
print('Mean protein intensity across the whole image:', red.mean())
print('Mean protein intensity near the cilia:', mean)

# Inspect the selected regions alongside the cilia image.
plt.figure(figsize=(12,4))
for panel, image, title in [(1, green, 'Cilia channel'),
                            (2, segm, 'Thresholded cilia'),
                            (3, msk, 'Expanded measurement region')]:
    plt.subplot(1,3,panel)
    plt.imshow(image)
    plt.title(title)
plt.tight_layout()
plt.show()

# Select no pixels: expected sum 0, area 0, and undefined mean.
print('No selected pixels; expected (0, 0, nan):',
      measure_near_cilia(green, red, green.max(), chosenRadius))

# Reuse the same function with a different radius, without editing its body.
largerRadius = chosenRadius + 5
largerTotal, largerArea, largerMean = measure_near_cilia(green, red, chosenThreshold, largerRadius)
print('Original radius:', chosenRadius, 'Larger radius:', largerRadius)
print('Intensity sums:', total, largerTotal)
print('Mask areas:', area, largerArea)
print('Mean intensities:', mean, largerMean)

**Reflect:** Which conditions from 4b are supported? What remains uncertain? Can you account for the steps and the effect of changing radius?

**Reference provenance:** Author-written example; not a recorded LLM response.

## 4.5 Evaluate the co-created result
- **Quality:** does the evidence support the intended measurement?
- **Reuse:** can a colleague apply it to another supported image pair?
- **Maintenance:** can they understand and change it without accidentally changing the measurement?

Saved decisions, settings, example results, and limitations support all three.

**Assignment 4d:** Review the plan, quality conditions, and code. Identify one strength, one limitation, and what a colleague needs to reuse or maintain the analysis.

**Predict:** If a colleague obtains a different mean intensity, which saved information would help diagnose why?

**Example quality review**

- **Strength:** the agreed plan, visible region, and base-case comparison make the calculation inspectable.
- **Limitation:** numerical agreement does not validate the threshold or distinguish protein signal from background.
- **Handover:** supply the notebook, images, settings, software versions, example outputs, and known limitations. A colleague should reproduce these before adapting the analysis.

**Reflect:** Where was your judgement essential? Do you understand the result well enough to take responsibility for it, and what evidence could a colleague use to assess it?